In [7]:
import torch
from torch import utils
import numpy as np
import torchvision as tv
from torchvision.models import vgg16 as VGG
from torchvision.datasets import Imagenette
import torchvision.transforms as tfs
import torch.nn as nn
from torchsummary import summary
from sklearn.model_selection import train_test_split
from pathlib import Path
import pickle
import os
import glob
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
transform = tfs.Compose([tfs.ToTensor(), tfs.Resize((320, 320))])
# Change download to 'True' if you don't have the dataset downloaded to your machine
data = Imagenette(root="./data", download=False, transform=transform)
d2 = Imagenette(root="./data")

0.7%

In [ ]:
len(data)

## Dr. Meni PEEK functions.py
[Linkkk](https://github.com/Kenzie-Meni/PEEK/blob/main/peek/functions.py)

I did change parts of it to make it work how I wanted it and such

In [8]:
class VGG16FeatureExtractor(torch.nn.Module):
    def __init__(self, weights='DEFAULT'):
        super(VGG16FeatureExtractor, self).__init__()
        self.vgg16 = tv.models.vgg16(weights=weights).features
        # Automatically collect indices of all convolutional layers
        self.conv_layers = [i for i, layer in enumerate(self.vgg16) if isinstance(layer, torch.nn.Conv2d)]

    def forward(self, x):
        features = []
        for layer_index, layer in enumerate(self.vgg16):
            x = layer(x)
            if layer_index in self.conv_layers:
                features.append(x)
        return features

    def load_image(self, image_path):
        # Load an image and transform it to the format required by VGG16
        transform = tfs.Compose([
            tfs.Resize(256),
            tfs.CenterCrop(224),
            tfs.ToTensor(),
            tfs.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        image = Image.open(image_path)
        image = transform(image).unsqueeze(0)  # Add batch dimension
        return image

    def save_features(self, frame_folder):
        _, save_folder = frame_folder.split('/')
        feature_folder = f'feature_maps/{save_folder}'

        Path(feature_folder).mkdir(parents=True, exist_ok=True)
        
        # Added ./ infront of frame_folder to make it local
        image_filepaths = sorted(glob.glob(f'./{frame_folder}/*'))

        for image_path in image_filepaths:
            
            input_tensor = self.load_image(image_path)
            
            with torch.no_grad():
                features = self.forward(input_tensor)

            # Create a base filename for saving features without the original extension
            base_filename = os.path.split(image_path)[-1].split('.')[0]
            
            filename = f'feature_maps/{save_folder}/{base_filename}.pkl'
            with open(filename, "wb") as f:
                pickle.dump([feature for feature in features], f)
            print(f"Saved all features to {filename}")

In [9]:
# Create Feature Extraction
features = VGG16FeatureExtractor()

# Load some images and get feature maps for them
features.save_features("/test1")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\1738a/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100.0%


Saved all features to feature_maps/test1/ILSVRC2012_val_00000293.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00002138.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00003014.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00006697.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00007197.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00009346.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00009379.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00009396.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00010306.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00011233.pkl
Saved all features to feature_maps/test1/ILSVRC2012_val_00011993.pkl


In [ ]:
# Get feature maps from an image 
path = "./feature_maps/test1/ILSVRC2012_val_00000293.pkl"
image = None

# UNPICKLE
with open (path, 'rb') as f:
    image = pickle.load(f)


In [ ]:
for layer in image:
    print(layer[0].size())

In [ ]:
for i in range(len(image[7][0])-5, len(image[7][0])):
    plt.imshow(tfs.functional.to_pil_image(image[7][0][i]))
    plt.show(0)

In [ ]:
# Tensors to image
tfs.functional.to_pil_image(image[2][0][2])

# I PPMI NOW

In [ ]:
import importlib
import fmppmi as ppmi
importlib.reload(ppmi)

In [ ]:
im1 = image[7][0][0]
im2 = image[7][0][1]

In [ ]:
pp1 = ppmi.fmppmi(im1, im2)
pp2 = ppmi.fmppmi(im2, im1)

In [ ]:
plt.imshow(pp1)
plt.show()

In [ ]:
plt.imshow(pp2)
plt.show()

In [ ]:
im3 = image[7][0][-2]
im4 = image[7][0][-1]

In [ ]:
v = ppmi.fmppmi(im3, im4)
b = ppmi.fmppmi(im4, im3)

In [ ]:
im5 = image[2][0][3]
im6 = image[2][0][2]

In [ ]:
print(torch.max(im5))
print(torch.div(im5, torch.max(im5)))

In [ ]:
plt.imshow(im5)

In [ ]:
pp5 = ppmi.fmppmi(im5, im6)
pp6 = ppmi.fmppmi(im6, im5)

In [ ]:
plt.imshow(pp5)
plt.show()

In [ ]:
plt.imshow(pp6)
plt.show()

In [ ]:
plt.imshow(im6)